In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import pickle
import matplotlib.pyplot as plt

In [2]:
# Use GPU if it is avaialable
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [3]:
df = pd.read_csv("Climate_data.csv")

data = df.drop("Date Time" , axis=1)
column_label = data.columns.tolist()
data = torch.tensor(data.values , dtype=torch.float32)

mean = data.mean(dim=0)
std = data.std(dim=0)

data = (data - mean)/std

In [6]:
print(len(data))

420551


In [12]:
data_X = []
data_y = []

for i in range(0,len(data)-(72*6*10)-1,(72*6*10)):
    data_X.append(data[i:i+(72*6*10)])
    data_y.append(data[i+(72*6*10)])
data_X = torch.stack(data_X)
data_y = torch.stack(data_y)

In [ ]:
class Encoder_Layer(nn.Module):
    def __init__(self , hidden_dim , num_heads):
        super().__init__()
        self.hidden_dim = hidden_dim 
        self.num_heads = num_heads
        self.head_dim = hidden_dim//num_heads

        # For Query Key and Values
        self.query = nn.Linear(in_features=hidden_dim , out_features=hidden_dim , bias=False)
        self.key = nn.Linear(in_features=hidden_dim , out_features=hidden_dim , bias=False)
        self.value = nn.Linear(in_features=hidden_dim , out_features=hidden_dim , bias=False)

        # Attention combining all the other attentions
        self.attention = nn.Linear(in_features=hidden_dim , out_features=hidden_dim)

        # Layer Normalisation
        self.layer_norm_1 = nn.LayerNorm(hidden_dim)
        self.layer_norm_2 = nn.LayerNorm(hidden_dim)

        # Feed Forward Network ffn
        self.ffn = nn.Sequential(
            nn.Linear(in_features=hidden_dim , out_features=hidden_dim*4),
            nn.ReLU(),
            nn.Linear(in_features=hidden_dim*4 , out_features=hidden_dim)
        )


    def forward(self , x):
        batch_size = x.shape[0]
        seq_len = x.shape[1]
        embedding = x

        # Querry , Key and Values
        Q = self.query(embedding)
        K = self.key(embedding)
        V = self.value(embedding)
        Q = Q.reshape(batch_size , seq_len , self.num_heads , self.head_dim).transpose(1,2)
        K = K.reshape(batch_size , seq_len , self.num_heads , self.head_dim).transpose(1,2)
        V = V.reshape(batch_size , seq_len , self.num_heads , self.head_dim).transpose(1,2)
        
        # Total Attention
        score = Q @ K.transpose(-2,-1)
        attention = torch.softmax(score/(self.head_dim**0.5) , dim=-1) @ V        
        attention = attention.transpose(1,2)
        attention = attention.reshape(batch_size , seq_len , self.hidden_dim)
        attention = self.attention(attention)

        # Feed Forward network and layer norm
        embedding = embedding + attention
        embedding = self.layer_norm_1(embedding)
        ffn_out = self.ffn(embedding)
        embedding = embedding + ffn_out
        embedding = self.layer_norm_2(embedding)

        # Returning the contextualised embeddings
        return embedding
    


class Encoder(nn.Module):
    def __init__(self , hidden_dim , num_heads):
        super().__init__()
        
        # Initially Convert the given vector into hidden dimension
        self.initial_embedding = nn.Linear(in_features=14 , out_features=hidden_dim)

        # Encoding layer
        self.layer_1 = Encoder_Layer(hidden_dim , num_heads)
        self.layer_2 = Encoder_Layer(hidden_dim , num_heads)

        # FInal Predictor
        self.final = nn.Linear(in_features=hidden_dim , out_features=14)

    def forward(self , x):
        # Initial Embedding 
        embedding = self.initial_embedding(x)

        # Positional Encoding
        positional_encoding = ...
        embedding = embedding + positional_encoding

        # Passing through embedding layers
        embedding = self.layer_1(embedding)
        embedding = self.layer_2(embedding)

        # Use last time as it has all the context from previous one also to predict next day weather
        y_pred = self.final(embedding[: , -1 , :])

        return y_pred

In [ ]:
# 1 - Main Query ko hidden dimension ka bana ke reshape kar do faster rahega
# 2 - ye ek encoder layer hai jo stack karni hai isiliye alag se me embedding karao intial aur fir isme input daalo
# 3 - Positional encoding bhi sirf ek hi baar judegi so use bhi dusre wale me rakh dena
# 4 - mean pooling try karo dekho kaunsa acha hai